# **1. Environment Setup**

In [1]:
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import numpy as np
import math
import os

2026-03-26 15:09:36.183516: W tensorflow/stream_executor/platform/default/dso_loader.cc:60] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-12.8/lib64:
2026-03-26 15:09:36.183530: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

## **1.1. Configure Default Paths**

In [ ]:
MODELS_DIR = "models/"

In [ ]:
if not os.path.exists(MODELS_DIR):
    os.mkdir(MODELS_DIR)

In [ ]:
# Define paths to model files

MODEL_TF = MODELS_DIR + "model"
MODEL_NO_QUANT_TFLITE = MODELS_DIR + "model_no_quant.tflite"
MODEL_TFLITE = MODELS_DIR + "model.tflite"
MODEL_TFLITE_MICRO = MODELS_DIR + "model.cc"

## **1.2. Configure Random Seeds**

In [ ]:
seed = 1

tf.random.set_seed(seed)
np.random.seed(seed)

# **2. Dataset Construction**

## **2.1. Data Generation**

In [ ]:
# Number of sample datapoints

NUM_SAMPLES = 1000

In [ ]:
# Generate a uniformly distributed set of random numbers in the range from
# 0 to 2π, which covers a complete sine wave oscillation

x_values = np.random.uniform(low = 0, high = 2 * math.pi, size = NUM_SAMPLES)

In [ ]:
# Shuffle the values to guarantee they are not in order

np.random.shuffle(x_values)

In [ ]:
# Calculate the corresponding sine values

y_values = np.sin(x_values).astype(np.float32)

In [ ]:
# Plot the data using blue dots

plt.plot(x_values, y_values, "b.")

plt.title("GENERATED DATA POINTS")
plt.xlabel("X")
plt.ylabel("Y")

plt.show()

## **2.2. Adding Noise**

In [ ]:
# Add a small random number to each y value

y_values += 0.1 * np.random.randn(*y_values.shape)

In [ ]:
# Plot the data using blue dots

plt.plot(x_values, y_values, "b.")

plt.title("GENERATED DATA POINTS WITH NOISE")
plt.xlabel("X")
plt.ylabel("Y")

plt.show()

## **2.3. Split the Data**

In [ ]:
# 60% of the data for training and 20% for testing. The remaining 20%
# will be used for validation

TRAIN_SPLIT =  int(0.6 * NUM_SAMPLES)
TEST_SPLIT = int(0.2 * NUM_SAMPLES + TRAIN_SPLIT)

In [ ]:
# Chop the data into train, test and validation sets using the indices

x_train, x_test, x_validate = np.split(x_values, [TRAIN_SPLIT, TEST_SPLIT])
y_train, y_test, y_validate = np.split(y_values, [TRAIN_SPLIT, TEST_SPLIT])

In [ ]:
# Double check that the splits add up correctly

assert (x_train.size + x_validate.size + x_test.size) ==  NUM_SAMPLES

In [ ]:
# Plot the data in each partition in different colors

plt.plot(x_train, y_train, "b.", label = "Train")
plt.plot(x_test, y_test, "r.", label = "Test")
plt.plot(x_validate, y_validate, "y.", label = "Validation")

plt.title("DATA SPLIT")

plt.xlabel("X")
plt.ylabel("Y")

plt.legend()

plt.show()

# **3. Model Training**

## **3.1. Design the Model**

In [ ]:
# Use Keras to create a simple model architecture

model_1 = tf.keras.Sequential([

    # Define the input shape explicitly as an Input object
    # For a sine wave, we take a single scalar value as input
    keras.layers.Input(shape=(1,), name = "input_layer"), 
    
    # First layer with 8 neurons and "relu" activation function
    keras.layers.Dense(8, activation = "relu", name = "hidden_layer"),
    
    # Final layer is a single neuron to output a single predicted value
    keras.layers.Dense(1, name = "output_layer")
])

In [ ]:
# Compile the model using the "adam" optimizer and "mse" loss function for regression

model_1.compile(
    optimizer = "adam", 
    loss = "mse", 
    metrics = ["mae"]
)

In [ ]:
# Display the model"s architecture

model_1.summary()

## **3.2. Train the Model**

In [ ]:
# Train the model on the training data while validating on the validation set

history_1 = model_1.fit(x_train, y_train, epochs = 500, batch_size = 64, validation_data = (x_validate, y_validate))

## **3.3. Plot Training Metrics**

### **3.3.1. Loss (MSE)**

In [ ]:
# Draw a graph of the loss

train_loss = history_1.history["loss"]
val_loss = history_1.history["val_loss"]

epochs = range(1, len(train_loss) + 1)

In [ ]:
plt.plot(epochs, train_loss, "g.", label = "Training loss")
plt.plot(epochs, val_loss, "b.", label="Validation loss")

plt.title("TRAINING AND VALIDATION LOSS")
plt.xlabel("EPOCHS")
plt.ylabel("LOSS")

plt.legend()

plt.show()

In [ ]:
# Exclude the first few epochs so the graph is easier to read

SKIP = 50

In [ ]:
plt.plot(epochs[SKIP:], train_loss[SKIP:], "g.", label = "Training loss")
plt.plot(epochs[SKIP:], val_loss[SKIP:], "b.", label = "Validation loss")

plt.title("TRAINING AND VALIDATION LOSS")
plt.xlabel("EPOCHS")
plt.ylabel("LOSS")

plt.legend()

plt.show()

### **3.3.2. Mean Absolute Error (MAE)**

In [ ]:
# Draw a graph of mean absolute error, which is another way of
# measuring the amount of error in the prediction

train_mae = history_1.history["mae"]
val_mae = history_1.history["val_mae"]

In [ ]:
plt.plot(epochs[SKIP:], train_mae[SKIP:], "g.", label = "Training MAE")
plt.plot(epochs[SKIP:], val_mae[SKIP:], "b.", label = "Validation MAE")

plt.title("TRAINING AND VALIDATION MAE")
plt.xlabel("EPOCHS")
plt.ylabel("MAE")

plt.legend()

plt.show()

### **3.3.3. Actual vs Predicted Output**

In [ ]:
# Calculate and print the loss on the test dataset
test_loss, test_mae = model_1.evaluate(x_test, y_test)

# Make predictions based on the test dataset
y_test_pred = model_1.predict(x_test)

In [ ]:
# Graph the predictions against the actual values

plt.plot(x_test, y_test, "b.", label = "Actual values")
plt.plot(x_test, y_test_pred, "r.", label = "TF predictions")

plt.title("COMPARISON OF PREDICTIONS AND ACTUAL VALUES")

plt.legend()

plt.show()

# **4. Training a Larger Model**

## **4.1. Design the Model**

In [ ]:
# Use Keras to create a simple model architecture

model_2 = tf.keras.Sequential([

    # Define the input shape explicitly as an Input object
    # For a sine wave, we take a single scalar value as input
    keras.layers.Input(shape=(1,), name = "input_layer"), 
    
    # First layer with 16 neurons and "relu" activation function
    keras.layers.Dense(16, activation = "relu", name = "hidden_layer_1"),

    # Second layer with 16 neurons and "relu" activation function
    keras.layers.Dense(16, activation = "relu", name = "hidden_layer_2"),
    
    # Final layer is a single neuron to output a single predicted value
    keras.layers.Dense(1, name = "output_layer")
])

In [ ]:
# Compile the model using the "adam" optimizer and "mse" loss function for regression

model_2.compile(
    optimizer = "adam", 
    loss = "mse", 
    metrics = ["mae"]
)

In [ ]:
# Display the model"s architecture

model_2.summary()

## **4.2. Train the Model**

In [ ]:
# Train the model on the training data while validating on the validation set

history_2 = model_2.fit(x_train, y_train, epochs = 500, batch_size = 64, validation_data = (x_validate, y_validate))

## **4.3. Plot Training Metrics**

### **4.3.1. Loss (MSE)**

In [ ]:
# Draw a graph of the loss

train_loss = history_2.history["loss"]
val_loss = history_2.history["val_loss"]

epochs = range(1, len(train_loss) + 1)

In [ ]:
plt.plot(epochs, train_loss, "g.", label = "Training loss")
plt.plot(epochs, val_loss, "b.", label="Validation loss")

plt.title("TRAINING AND VALIDATION LOSS")
plt.xlabel("EPOCHS")
plt.ylabel("LOSS")

plt.legend()

plt.show()

In [ ]:
# Exclude the first few epochs so the graph is easier to read

SKIP = 50

In [ ]:
plt.plot(epochs[SKIP:], train_loss[SKIP:], "g.", label = "Training loss")
plt.plot(epochs[SKIP:], val_loss[SKIP:], "b.", label = "Validation loss")

plt.title("TRAINING AND VALIDATION LOSS")
plt.xlabel("EPOCHS")
plt.ylabel("LOSS")

plt.legend()

plt.show()

### **4.3.2. Mean Absolute Error (MAE)**

In [ ]:
# Draw a graph of mean absolute error, which is another way of
# measuring the amount of error in the prediction

train_mae = history_2.history["mae"]
val_mae = history_2.history["val_mae"]

In [ ]:
plt.plot(epochs[SKIP:], train_mae[SKIP:], "g.", label = "Training MAE")
plt.plot(epochs[SKIP:], val_mae[SKIP:], "b.", label = "Validation MAE")

plt.title("TRAINING AND VALIDATION MAE")
plt.xlabel("EPOCHS")
plt.ylabel("MAE")

plt.legend()

plt.show()

### **4.3.3. Actual vs Predicted Output**

In [ ]:
# Calculate and print the loss on the test dataset
test_loss, test_mae = model_2.evaluate(x_test, y_test)

# Make predictions based on the test dataset
y_test_pred = model_2.predict(x_test)

In [ ]:
# Graph the predictions against the actual values

plt.plot(x_test, y_test, "b.", label = "Actual values")
plt.plot(x_test, y_test_pred, "r.", label = "TF predictions")

plt.title("COMPARISON OF PREDICTIONS AND ACTUAL VALUES")

plt.legend()

plt.show()

In [ ]:
# Save the best model (model 2) to disk
model_2.export(MODEL_TF)

# **5. Generate a TensorFlow Lite Model**

## **5.1. Generate Models with or without Quantization**

In [ ]:
# Convert the model to the TensorFlow Lite format without quantization

converter = tf.lite.TFLiteConverter.from_saved_model(MODEL_TF)
model_no_quant_tflite = converter.convert()

In [ ]:
# Save the model to disk

open(MODEL_NO_QUANT_TFLITE, "wb").write(model_no_quant_tflite)

In [ ]:
# Convert the model to the TensorFlow Lite format with quantization

def representative_dataset():
  for i in range(500):
    yield([x_train[i].reshape(1, 1).astype(np.float32)])

In [ ]:
# Set the optimization flag

converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Enforce integer only quantization

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Provide a representative dataset to ensure we quantize correctly

converter.representative_dataset = representative_dataset
model_tflite = converter.convert()

In [ ]:
# Save the model to disk
open(MODEL_TFLITE, "wb").write(model_tflite)

## **5.2. Compare Model Performance**

In [ ]:
def predict_tflite(tflite_model, x_test):
  # Prepare the test data
  x_test_ = x_test.copy()
  x_test_ = x_test_.reshape((x_test.size, 1))
  x_test_ = x_test_.astype(np.float32)

  # Initialize the TFLite interpreter
  interpreter = tf.lite.Interpreter(model_content=tflite_model,
                                    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_REF)
  interpreter.allocate_tensors()

  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  # If required, quantize the input layer (from float to integer)
  input_scale, input_zero_point = input_details["quantization"]
  if (input_scale, input_zero_point) != (0.0, 0):
    x_test_ = x_test_ / input_scale + input_zero_point
    x_test_ = x_test_.astype(input_details["dtype"])
  
  # Invoke the interpreter
  y_pred = np.empty(x_test_.size, dtype=output_details["dtype"])
  for i in range(len(x_test_)):
    interpreter.set_tensor(input_details["index"], [x_test_[i]])
    interpreter.invoke()
    y_pred[i] = interpreter.get_tensor(output_details["index"])[0]
  
  # If required, dequantized the output layer (from integer to float)
  output_scale, output_zero_point = output_details["quantization"]
  if (output_scale, output_zero_point) != (0.0, 0):
    y_pred = y_pred.astype(np.float32)
    y_pred = (y_pred - output_zero_point) * output_scale

  return y_pred

def evaluate_tflite(tflite_model, x_test, y_true):
  global model
  y_pred = predict_tflite(tflite_model, x_test)
  loss_function = tf.keras.losses.get(model.loss)
  loss = loss_function(y_true, y_pred).numpy()
  return loss

### **5.2.1. Predictions**

In [ ]:
# Calculate predictions
y_test_pred_tf = model_2.predict(x_test)
y_test_pred_no_quant_tflite = predict_tflite(model_no_quant_tflite, x_test)
y_test_pred_tflite = predict_tflite(model_tflite, x_test)